# 06 — Modeling: Advanced Classifiers

Trains Random Forest, XGBoost, and SVM-RBF. Verifies XGBoost test AUC ∈ [0.88, 0.94] and produces a full 7-model comparison table.

In [ ]:
import sys, os
sys.path.insert(0, '../src')

import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import json, joblib
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 110

RANDOM_STATE = 42
os.makedirs('../models',  exist_ok=True)
os.makedirs('../figures', exist_ok=True)


## 1 · Load Features and Split

In [ ]:
import features, splits

df = pd.read_parquet('../data/modeling_table.parquet')
train_mask, val_mask, test_mask = splits.temporal_split(df, train_frac=0.60, val_frac=0.20)

X, feature_names = features.build_feature_matrix(df, train_mask)
y = df['y_primary'].values

X_train, y_train = X[train_mask], y[train_mask]
X_val,   y_val   = X[val_mask],   y[val_mask]
X_test,  y_test  = X[test_mask],  y[test_mask]

X_trainval = np.vstack([X_train, X_val])
y_trainval  = np.concatenate([y_train, y_val])

print(f"Train+Val: {X_trainval.shape}  |  Test: {X_test.shape}")


## 2 · Evaluation Helper

In [ ]:
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    accuracy_score, f1_score, brier_score_loss,
    roc_curve,
)
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5)
results_adv = {}

def evaluate(name, model, X_te, y_te, proba=True):
    if proba:
        probs = model.predict_proba(X_te)[:, 1]
    else:
        probs = model.decision_function(X_te)
        probs = (probs - probs.min()) / (probs.max() - probs.min())
    preds = model.predict(X_te)
    metrics = {
        'ROC-AUC' : round(roc_auc_score(y_te, probs), 4),
        'PR-AUC'  : round(average_precision_score(y_te, probs), 4),
        'Accuracy': round(accuracy_score(y_te, preds), 4),
        'F1'      : round(f1_score(y_te, preds), 4),
        'Brier'   : round(brier_score_loss(y_te, probs), 4),
    }
    results_adv[name] = metrics
    for k, v in metrics.items():
        print(f"  {k:9s}: {v}")
    return probs


## 3 · Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_params = {
    'n_estimators'    : [200, 400, 600],
    'max_depth'       : [8, 12, 16, None],
    'min_samples_leaf': [5, 10, 20],
    'max_features'    : ['sqrt', 0.3, 0.5],
    'class_weight'    : ['balanced', None],
}

rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    rf_params, n_iter=20, cv=tscv,
    scoring='roc_auc', n_jobs=1,
    random_state=RANDOM_STATE, verbose=1,
)
rf_search.fit(X_train, y_train)
print("Best params:", rf_search.best_params_)

rf_best = rf_search.best_estimator_
rf_best.set_params(n_jobs=-1)
rf_best.fit(X_trainval, y_trainval)

print("\nTest-set performance:")
rf_probs = evaluate('RandomForest', rf_best, X_test, y_test)

joblib.dump(rf_best, '../models/random_forest.pkl')
print("Saved → ../models/random_forest.pkl")


## 4 · XGBoost

In [ ]:
from xgboost import XGBClassifier

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_params = {
    'n_estimators'    : [300, 500, 700],
    'max_depth'       : [4, 5, 6, 7],
    'learning_rate'   : [0.01, 0.05, 0.1],
    'subsample'       : [0.7, 0.8, 0.9],
    'colsample_bytree': [0.6, 0.7, 0.8],
    'min_child_weight': [1, 5, 10],
    'reg_alpha'       : [0, 0.1, 1.0],
    'reg_lambda'      : [1.0, 5.0, 10.0],
}

xgb_search = RandomizedSearchCV(
    XGBClassifier(
        random_state=RANDOM_STATE,
        scale_pos_weight=scale_pos_weight,
        eval_metric='auc',
        verbosity=0,
        n_jobs=-1,
    ),
    xgb_params, n_iter=20, cv=tscv,
    scoring='roc_auc', n_jobs=1,
    random_state=RANDOM_STATE, verbose=1,
)
xgb_search.fit(X_train, y_train)
print("Best params:", xgb_search.best_params_)

xgb_best = xgb_search.best_estimator_
xgb_best.fit(X_trainval, y_trainval)

print("\nTest-set performance:")
xgb_probs = evaluate('XGBoost', xgb_best, X_test, y_test)

joblib.dump(xgb_best, '../models/xgboost.pkl')
print("Saved → ../models/xgboost.pkl")


## 5 · XGBoost AUC Assertion

In [ ]:
xgb_auc = results_adv['XGBoost']['ROC-AUC']
assert 0.88 <= xgb_auc <= 0.94, (
    f"XGBoost test AUC {xgb_auc:.4f} is outside expected range [0.88, 0.94]. "
    "Recheck hyperparameters or feature engineering."
)
print(f"XGBoost AUC assertion PASSED: {xgb_auc:.4f} ∈ [0.88, 0.94] ✓")


## 6 · SVM-RBF

> **Note:** SVM does not scale to 750 k rows. We subsample to **30 k** training examples and verify that doubling to **60 k** changes AUC by < 0.005 (confirming the subsample is representative).

In [ ]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

scaler_svm = joblib.load('../models/standard_scaler.pkl')
X_train_s   = scaler_svm.transform(X_train)
X_trainval_s = scaler_svm.transform(X_trainval)
X_test_s     = scaler_svm.transform(X_test)

RNG = np.random.default_rng(RANDOM_STATE)

# 30 k subsample
idx_30k = RNG.choice(len(X_train_s), size=30_000, replace=False)
X_30k, y_30k = X_train_s[idx_30k], y_train[idx_30k]

svm_params = {
    'C'    : [0.1, 1.0, 10.0, 100.0],
    'gamma': ['scale', 'auto', 0.001, 0.01],
}

svm_search_30k = RandomizedSearchCV(
    SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE, cache_size=500),
    svm_params, n_iter=10, cv=3,
    scoring='roc_auc', n_jobs=-1,
    random_state=RANDOM_STATE, verbose=1,
)
svm_search_30k.fit(X_30k, y_30k)
print("Best params (30 k):", svm_search_30k.best_params_)

svm_30k = svm_search_30k.best_estimator_
auc_30k = roc_auc_score(y_test, svm_30k.predict_proba(X_test_s)[:, 1])
print(f"SVM AUC @ 30 k: {auc_30k:.4f}")


In [ ]:
# 60 k subsample — stability check
idx_60k = RNG.choice(len(X_train_s), size=60_000, replace=False)
X_60k, y_60k = X_train_s[idx_60k], y_train[idx_60k]

svm_60k = SVC(
    kernel='rbf', probability=True,
    random_state=RANDOM_STATE, cache_size=500,
    **svm_search_30k.best_params_,
)
svm_60k.fit(X_60k, y_60k)
auc_60k = roc_auc_score(y_test, svm_60k.predict_proba(X_test_s)[:, 1])
print(f"SVM AUC @ 60 k: {auc_60k:.4f}")

delta_svm = abs(auc_60k - auc_30k)
print(f"AUC delta (60 k − 30 k): {delta_svm:+.4f}")
assert delta_svm < 0.005, (
    f"SVM subsample NOT stable: AUC change {delta_svm:.4f} >= 0.005. "
    "Increase subsample size."
)
print("SVM subsample stability PASSED ✓")


In [ ]:
# Final SVM: train on full train+val scaled (subsample 60 k from it)
idx_tv = RNG.choice(len(X_trainval_s), size=min(60_000, len(X_trainval_s)), replace=False)
svm_final = SVC(
    kernel='rbf', probability=True,
    random_state=RANDOM_STATE, cache_size=500,
    **svm_search_30k.best_params_,
)
svm_final.fit(X_trainval_s[idx_tv], y_trainval[idx_tv])

print("\nTest-set performance:")
svm_probs = evaluate('SVM-RBF', svm_final, X_test_s, y_test)

results_adv['SVM-RBF']['note'] = f'30k→60k AUC delta={delta_svm:.4f}'
joblib.dump(svm_final, '../models/svm_rbf.pkl')
print("Saved → ../models/svm_rbf.pkl")


## 7 · ROC Curves — Advanced Models

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for name, probs in [
    ('RandomForest', rf_probs),
    ('XGBoost',      xgb_probs),
    ('SVM-RBF',      svm_probs),
]:
    fpr, tpr, _ = roc_curve(y_test, probs)
    auc = results_adv[name]['ROC-AUC']
    ax.plot(fpr, tpr, label=f"{name}  (AUC = {auc:.4f})")

ax.plot([0, 1], [0, 1], 'k--', linewidth=0.8, label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — Advanced Models (Test Set)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('../figures/06_roc_curves_advanced.png', bbox_inches='tight')
plt.show()


## 8 · Full 7-Model Comparison Table

In [ ]:
# Load baseline results saved from notebook 05
try:
    with open('../data/baseline_results.json') as f:
        results_baseline = json.load(f)
except FileNotFoundError:
    print("baseline_results.json not found — run notebook 05 first.")
    results_baseline = {}

all_results = {**results_baseline, **results_adv}

comparison = pd.DataFrame(all_results).T[['ROC-AUC', 'PR-AUC', 'Accuracy', 'F1', 'Brier']]
comparison.index.name = 'Model'
comparison = comparison.sort_values('ROC-AUC', ascending=False)
print(comparison.to_string())
comparison


In [ ]:
# Persist advanced results for downstream notebooks
with open('../data/advanced_results.json', 'w') as f:
    json.dump({k: {m: v for m, v in metrics.items() if m != 'note'}
               for k, metrics in results_adv.items()}, f)
print("Saved → ../data/advanced_results.json")
